# Libaries

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from src.benchmarks import (
    BenchmarkConfig,
    RidgeConfig,
    PairDatasetBuilder,
    PairwiseElasticityPipeline,
    RegularizedElasticityPipeline,
    BootstrapSummarizer,
)
from src.dominick import DominickDataLoader
from src.utils import TemporalSplitter, BlockBootstrapSampler

In [2]:
TRAIN_FRAC = 0.8
N_FOLDS = 5
N_BOOTSTRAP = 20
SELECTED_UPCS = [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]
config = BenchmarkConfig()

# Loader

In [3]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
df = df[df["upc_code"].isin(SELECTED_UPCS)].copy()

pair_builder = PairDatasetBuilder(control_cols=config.control_cols)
pair_df = pair_builder.build(df)

print(f"Dataset: {df.shape}, Pairs: {pair_df.shape}")

Dataset: (73209, 44), Pairs: (225004, 24)


In [4]:
splitter = TemporalSplitter(week_col="week_id")
pipeline = PairwiseElasticityPipeline(config)
sampler = BlockBootstrapSampler(week_col="week_id", block_size=4, rng=np.random.default_rng(42))
summarizer = BootstrapSummarizer()

In [5]:
train_df, val_df = splitter.single_split(pair_df, train_frac=TRAIN_FRAC)
train_weeks = sorted(train_df["week_id"].unique())

# K-fold

In [6]:
fold_results, fold_predictions = [], []
for fold_idx, (train_fold, val_fold) in enumerate(splitter.expanding_splits(pair_df, N_FOLDS)):
    print(f"Fold {fold_idx} of {N_FOLDS}")
    res, preds = pipeline.run(train_fold, val_fold)
    res["fold"] = fold_idx
    preds["fold"] = fold_idx
    fold_results.append(res)
    fold_predictions.append(preds)

all_folds = pd.concat(fold_results, ignore_index=True)
ok_folds = all_folds[all_folds["status"] == "ok"].copy()
ok_predictions = pd.concat(fold_predictions, ignore_index=True)
print(f"K-fold raw: {len(ok_folds)} rows")
print(f"K-fold predictions raw: {len(ok_predictions)} rows")

Fold 0 of 5
Fold 1 of 5
Fold 2 of 5
Fold 3 of 5
Fold 4 of 5
K-fold raw: 3944 rows
K-fold predictions raw: 103226 rows


# Bootstrap

In [7]:
bootstrap_results = []
for b in range(N_BOOTSTRAP):
    print(f"Bootstrap {b} of {N_BOOTSTRAP}")
    train_bs = sampler.sample(train_df, train_weeks)
    res, _ = pipeline.run(train_bs, val_df)  
    res["bootstrap_run"] = b
    bootstrap_results.append(res)

all_bootstrap = pd.concat(bootstrap_results, ignore_index=True)
ok_bs = all_bootstrap[all_bootstrap["status"] == "ok"].copy()
print(f"Bootstrap raw: {len(ok_bs)} rows")

Bootstrap 0 of 20
Bootstrap 1 of 20
Bootstrap 2 of 20
Bootstrap 3 of 20
Bootstrap 4 of 20
Bootstrap 5 of 20
Bootstrap 6 of 20
Bootstrap 7 of 20
Bootstrap 8 of 20
Bootstrap 9 of 20
Bootstrap 10 of 20
Bootstrap 11 of 20
Bootstrap 12 of 20
Bootstrap 13 of 20
Bootstrap 14 of 20
Bootstrap 15 of 20
Bootstrap 16 of 20
Bootstrap 17 of 20
Bootstrap 18 of 20
Bootstrap 19 of 20
Bootstrap raw: 10880 rows


# Summaries

In [8]:
bootstrap_summary = summarizer.summarize_raw(ok_bs)

print(f"K-fold raw: {len(ok_folds)}")
print(f"Bootstrap summary: {len(bootstrap_summary)}")

K-fold raw: 3944
Bootstrap summary: 544


# Display

In [9]:
display(ok_folds.head(10))
display(ok_folds[["own_elasticity", "cross_elasticity", "mae_val", "rmse_val", "r2_val"]].describe())
display(bootstrap_summary.head(10))
display(all_folds["status"].value_counts(dropna=False))
display(all_bootstrap["status"].value_counts(dropna=False))

,store_code,pair_id,upc_i,upc_j,status,n_train,n_val,own_elasticity,own_elasticity_ci_low,own_elasticity_ci_high,own_elasticity_p_value,cross_elasticity,cross_elasticity_ci_low,cross_elasticity_ci_high,cross_elasticity_p_value,mae_val,rmse_val,r2_val,fold
0,5,3410017306.0__7289000011.0,3410017306,7289000011,ok,49,21,-3.231876,-10.292528,3.828776,3.696468e-01,0.143843,-16.414304,16.701989,0.986416,0.500296,0.623773,-0.098164,0
1,5,3410017306.0__7289000011.0,7289000011,3410017306,ok,49,21,1.428486,-10.277905,13.134876,8.109762e-01,-3.832222,-9.843352,2.178908,0.211476,0.840478,1.009071,-1.963936,0
2,8,1820000784.0__3410010505.0,1820000784,3410010505,ok,140,30,-3.980905,-5.830347,-2.131462,2.456042e-05,-1.234048,-2.646545,0.178449,0.086832,0.538855,0.704073,0.245433,0
3,8,1820000784.0__3410010505.0,3410010505,1820000784,ok,140,30,-4.212563,-6.343453,-2.081673,1.067742e-04,-0.775208,-2.015348,0.464932,0.220512,0.573049,0.698324,-0.213861,0
4,8,1820000784.0__3410017306.0,1820000784,3410017306,ok,142,30,-4.399785,-6.211706,-2.587863,1.942971e-06,-0.111635,-2.372310,2.149041,0.922897,0.531910,0.654544,0.347862,0
5,8,1820000784.0__3410017306.0,3410017306,1820000784,ok,142,30,-6.894701,-8.926355,-4.863047,2.902981e-11,-0.363335,-1.464546,0.737876,0.517843,0.554172,0.690565,-0.873490,0
6,8,1820000784.0__7289000011.0,1820000784,7289000011,ok,101,20,-3.765933,-5.760757,-1.771109,2.154953e-04,-3.926900,-10.769218,2.915418,0.260653,0.589636,0.709794,0.256038,0
7,8,1820000784.0__7289000011.0,7289000011,1820000784,ok,101,20,-2.149943,-9.852645,5.552759,5.843402e-01,-0.882123,-2.169530,0.405285,0.179287,0.512074,0.611032,0.108886,0
8,8,3410010505.0__3410017306.0,3410010505,3410017306,ok,141,30,-4.753681,-7.001935,-2.505427,3.411155e-05,0.164301,-1.859324,2.187927,0.873564,0.659298,0.817720,-0.664425,0
9,8,3410010505.0__3410017306.0,3410017306,3410010505,ok,141,30,-6.853583,-8.891930,-4.815236,4.397279e-11,-1.169837,-2.322327,-0.017348,0.046650,0.405355,0.522281,-0.071645,0


,own_elasticity,cross_elasticity,mae_val,rmse_val,r2_val
count,3944.000000,3944.000000,3944.000000,3944.000000,3944.000000
mean,-3.979438,-0.296570,0.548205,0.676296,-0.625718
std,2.341376,1.388284,0.319506,0.366398,2.854190
min,-13.428422,-12.408779,0.159819,0.205145,-61.175994
25%,-5.403544,-0.807503,0.393818,0.496390,-0.507027
50%,-4.079622,-0.313359,0.486831,0.608772,-0.032410
75%,-2.603769,0.188139,0.598794,0.743724,0.293210
max,13.982406,15.694139,4.441895,4.968948,0.875174


,store_code,pair_id,upc_i,upc_j,own_elasticity_mean,own_elasticity_std,own_elasticity_ci_low,own_elasticity_ci_high,cross_elasticity_mean,cross_elasticity_std,cross_elasticity_ci_low,cross_elasticity_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,-3.943192,0.664774,-5.171571,-2.751409,-0.524214,0.594388,-1.651882,0.325129,0.502926,0.033820,0.670437,0.029474,0.067664,0.082727
1,8,1820000784.0__3410010505.0,3410010505,1820000784,-3.828953,0.471752,-4.743891,-3.000822,0.632335,0.545422,-0.512373,1.346856,0.656213,0.099905,0.795443,0.102688,-1.044767,0.546676
2,8,1820000784.0__7289000011.0,1820000784,7289000011,-4.105047,0.914318,-5.485459,-2.407024,-3.364364,1.133453,-5.393933,-1.619778,0.576561,0.040925,0.734651,0.047671,-0.121905,0.149046
3,8,1820000784.0__7289000011.0,7289000011,1820000784,-0.725216,1.909239,-3.409498,2.857046,-0.718401,0.540156,-1.551764,0.189615,1.269831,0.535784,1.432379,0.536141,-3.933260,3.502777
4,8,3410010505.0__7289000011.0,3410010505,7289000011,-3.601976,0.663150,-4.679613,-2.555741,-0.888785,0.793586,-2.529272,0.447748,0.624402,0.093542,0.760607,0.102791,-0.872383,0.509173
5,8,3410010505.0__7289000011.0,7289000011,3410010505,-1.160975,2.542112,-4.841089,3.652189,-0.416067,0.755284,-1.561561,0.966771,1.487511,0.733984,1.650938,0.729546,-5.856748,5.812704
6,9,1820000784.0__3410010505.0,1820000784,3410010505,-4.963919,0.766560,-6.565676,-3.679951,0.101413,0.809821,-1.168200,1.451967,0.864118,0.111557,1.049908,0.106082,-1.482864,0.516701
7,9,1820000784.0__3410010505.0,3410010505,1820000784,-2.945786,0.455627,-3.653389,-2.280265,0.385623,0.981470,-1.309460,2.195804,0.387388,0.042200,0.472885,0.044831,-0.136355,0.233235
8,9,1820000784.0__7289000011.0,1820000784,7289000011,-4.309412,0.839982,-5.845607,-2.806701,-0.254099,1.240187,-2.271599,1.781571,0.809601,0.094073,0.992559,0.090041,-1.214898,0.406498
9,9,1820000784.0__7289000011.0,7289000011,1820000784,-4.196879,2.921108,-9.744212,-0.392918,1.014799,0.430685,0.308489,1.666167,0.464500,0.064174,0.573278,0.067992,0.217371,0.192385


status
ok    3944
Name: count, dtype: int64

status
ok    10880
Name: count, dtype: int64

# Save

In [10]:
ok_folds.to_csv("../data/benchmark_kfold_raw.csv", index=False)
ok_predictions.to_csv("../data/benchmark_kfold_predictions_raw.csv", index=False)
ok_bs.to_csv("../data/benchmark_bootstrap_raw.csv", index=False)
bootstrap_summary.to_csv("../data/benchmark_elasticities_bootstrap_summary.csv", index=False)
print("Saved OK")

Saved OK


# Ridge Benchmark

In [11]:
ridge_config = RidgeConfig()
ridge_pipeline = RegularizedElasticityPipeline(ridge_config)

In [12]:
ridge_fold_results, ridge_fold_predictions = [], []
for fold_idx, (train_fold, val_fold) in enumerate(splitter.expanding_splits(pair_df, N_FOLDS)):
    print(f"[Ridge] Fold {fold_idx} of {N_FOLDS}")
    res, preds = ridge_pipeline.run(train_fold, val_fold)
    res["fold"] = fold_idx
    preds["fold"] = fold_idx
    ridge_fold_results.append(res)
    ridge_fold_predictions.append(preds)

ridge_all_folds = pd.concat(ridge_fold_results, ignore_index=True)
ridge_ok_folds = ridge_all_folds[ridge_all_folds["status"] == "ok"].copy()
ridge_ok_predictions = pd.concat(ridge_fold_predictions, ignore_index=True)
print(f"Ridge K-fold raw: {len(ridge_ok_folds)} rows")
print(f"Ridge K-fold predictions raw: {len(ridge_ok_predictions)} rows")

[Ridge] Fold 0 of 5
[Ridge] Fold 1 of 5
[Ridge] Fold 2 of 5
[Ridge] Fold 3 of 5
[Ridge] Fold 4 of 5
Ridge K-fold raw: 3944 rows
Ridge K-fold predictions raw: 103226 rows


In [13]:
ridge_bootstrap_results = []
for b in range(N_BOOTSTRAP):
    print(f"[Ridge] Bootstrap {b} of {N_BOOTSTRAP}")
    train_bs = sampler.sample(train_df, train_weeks)
    res, _ = ridge_pipeline.run(train_bs, val_df)   
    res["bootstrap_run"] = b
    ridge_bootstrap_results.append(res)

ridge_all_bootstrap = pd.concat(ridge_bootstrap_results, ignore_index=True)
ridge_ok_bs = ridge_all_bootstrap[ridge_all_bootstrap["status"] == "ok"].copy()
print(f"Ridge Bootstrap raw: {len(ridge_ok_bs)} rows")

[Ridge] Bootstrap 0 of 20
[Ridge] Bootstrap 1 of 20
[Ridge] Bootstrap 2 of 20
[Ridge] Bootstrap 3 of 20
[Ridge] Bootstrap 4 of 20
[Ridge] Bootstrap 5 of 20
[Ridge] Bootstrap 6 of 20
[Ridge] Bootstrap 7 of 20
[Ridge] Bootstrap 8 of 20
[Ridge] Bootstrap 9 of 20
[Ridge] Bootstrap 10 of 20
[Ridge] Bootstrap 11 of 20
[Ridge] Bootstrap 12 of 20
[Ridge] Bootstrap 13 of 20
[Ridge] Bootstrap 14 of 20
[Ridge] Bootstrap 15 of 20
[Ridge] Bootstrap 16 of 20
[Ridge] Bootstrap 17 of 20
[Ridge] Bootstrap 18 of 20
[Ridge] Bootstrap 19 of 20
Ridge Bootstrap raw: 10880 rows


In [14]:
ridge_bootstrap_summary = summarizer.summarize_raw(ridge_ok_bs)

ridge_ok_folds.to_csv("../data/benchmark_ridge_kfold_raw.csv", index=False)
ridge_ok_predictions.to_csv("../data/benchmark_ridge_kfold_predictions_raw.csv", index=False)
ridge_ok_bs.to_csv("../data/benchmark_ridge_bootstrap_raw.csv", index=False)
ridge_bootstrap_summary.to_csv("../data/benchmark_ridge_elasticities_bootstrap_summary.csv", index=False)
print("Saved Ridge: kfold_raw, kfold_predictions_raw, bootstrap_raw, bootstrap_summary")

Saved Ridge: kfold_raw, kfold_predictions_raw, bootstrap_raw, bootstrap_summary
